In [58]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
import pandas as pd

In [60]:
groq_api_key = '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'

In [59]:
df = pd.read_excel("C:/Users/Kannan T/Documents/CIF_Mapping_Sheet.xlsx")

In [61]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():

    content = f"""
Source Table: {row['Owner business Object']}
Source Attribute: {row['Owner Object Attribute']}
EDL Table: {row['Mapping Object Name (EDL)']}
EDL Attribute: {row['Mapping Object Attribute (EDL)']}
"""

    documents.append(
        Document(
            page_content=content))

In [62]:
print(df.head())

       Mapping Object Name (EDL) Mapping Object Attribute (EDL)  \
0  cif_notification_subscription                notification_id   
1  cif_notification_subscription                       activity   
2  cif_notification_subscription                    activity_cd   
3  cif_notification_subscription                  activity_desc   
4  cif_notification_subscription                       currency   

   Owner business Object Owner Object Attribute  
0  CIF_NOTIFCN_SUBSCRPTN        notification_id  
1  CIF_NOTIFCN_SUBSCRPTN               activity  
2  CIF_NOTIFCN_SUBSCRPTN            activity_cd  
3                    NaN                    NaN  
4  CIF_NOTIFCN_SUBSCRPTN           amount_crncy  


In [63]:
documents

[Document(metadata={}, page_content='\nSource Table: CIF_NOTIFCN_SUBSCRPTN\nSource Attribute: notification_id\nEDL Table: cif_notification_subscription\nEDL Attribute: notification_id\n'),
 Document(metadata={}, page_content='\nSource Table: CIF_NOTIFCN_SUBSCRPTN\nSource Attribute: activity\nEDL Table: cif_notification_subscription\nEDL Attribute: activity\n'),
 Document(metadata={}, page_content='\nSource Table: CIF_NOTIFCN_SUBSCRPTN\nSource Attribute: activity_cd\nEDL Table: cif_notification_subscription\nEDL Attribute: activity_cd\n'),
 Document(metadata={}, page_content='\nSource Table: nan\nSource Attribute: nan\nEDL Table: cif_notification_subscription\nEDL Attribute: activity_desc\n'),
 Document(metadata={}, page_content='\nSource Table: CIF_NOTIFCN_SUBSCRPTN\nSource Attribute: amount_crncy\nEDL Table: cif_notification_subscription\nEDL Attribute: currency\n'),
 Document(metadata={}, page_content='\nSource Table: CIF_NOTIFCN_SUBSCRPTN\nSource Attribute: amount_threshold\nEDL Tab

In [10]:
#loader = TextLoader("C:/Users/Kannan T/Documents/company.txt")
#documents = loader.load()

#print("Document loaded successfully")
#print("Number of documents:", len(documents))

In [64]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

In [65]:
chunks = text_splitter.split_documents(documents)
print("Number of chunks:", len(chunks))

Number of chunks: 844


In [66]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [67]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

In [68]:
# Create retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}  #How many documents/chunks should the retriever return? (Return the 2 most relevant chunks)
)

In [69]:
chat_history = []

In [70]:
prompt_template = """
You are a helpful AI assistant.

Answer the user's question using only the information provided in the context.

Context:
{context}

Previous Conversations:
{chat_history}

Question:
{question}

Instructions:
- Provide only the answer.
- Keep the answer concise and relevant.
- Do not make up information.
- If the context does not contain enough information, say:
  "The available data does not contain sufficient information to answer this question."
"""

In [71]:
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=[
        "context",
        "question",
        "chat_history"
    ]
)

In [72]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=groq_api_key
)

In [73]:
rewrite_prompt = PromptTemplate(
    template="""
Given the previous conversation and the user's latest question,
rewrite the latest question so that it can be understood independently.

Do not answer the question.
Only return the standalone question.

Previous conversation:
{chat_history}

Latest question:
{question}

Standalone question:
""",
    input_variables=["chat_history", "question"]
)

In [74]:
question_rewriter = (
    {
        "chat_history": lambda x: chat_history,
        "question": RunnablePassthrough()    #take is input question
    }
    | rewrite_prompt
    | llm              #llm.invoke(result from rewrite_prompt)
)

In [75]:
def ask_question(question):

    # Step 1: Rewrite the question using conversation history
    standalone_question = question_rewriter.invoke(
        question
    ).content

    # Step 2: Retrieve documents using rewritten question
    docs = retriever.invoke(
        standalone_question
    )

    # Step 3: Create context
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Step 4: Generate final answer
    prompt = PROMPT.format(
        context=context,
        chat_history=chat_history,
        question=question
    )

    response = llm.invoke(prompt)

    answer = response.content

    # Step 5: Update conversation history
    chat_history.append(
        ("user", question)
    )

    chat_history.append(
        ("assistant", answer)
    )

    return answer

In [17]:
                         USER
                          │
                          ▼
                   Latest Question
                          │
                          ▼
                 ┌─────────────────┐
                 │ Question         │
                 │ Rewriter         │
                 │ GPT-OSS-120B     │
                 └────────┬────────┘
                          │
                          ▼
                Standalone Question
                          │
                          ▼
                    Embedding Model
                  all-MiniLM-L6-v2
                          │
                          ▼
                        FAISS
                          │
                          ▼
                  Relevant Documents
                          │
                          ▼
                 ┌─────────────────┐
                 │ Final Prompt    │
                 │                 │
                 │ Context         │
                 │ History         │
                 │ Question        │
                 └────────┬────────┘
                          │
                          ▼
                    GPT-OSS-120B
                          │
                          ▼
                        Answer
                          │
                          ▼
                   Update History

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 18)

In [76]:
query = "What is the EDL attribute corresponding to amount_crncy in CIF_NOTIFCN_SUBSCRPTN?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)


--- Document 1 ---
Source Table: CIF_NOTIFCN_SUBSCRPTN
Source Attribute: amount_threshold
EDL Table: cif_notification_subscription
EDL Attribute: amount_above

--- Document 2 ---
Source Table: CIF_NOTIFCN_SUBSCRPTN
Source Attribute: amount_crncy
EDL Table: cif_notification_subscription
EDL Attribute: currency


In [77]:
while True:
    question = input("Ask a question")
    if question == "exit":
        break
    print(ask_question(question))

Ask a question What is the EDL attribute corresponding to amount_crncy in CIF_NOTIFCN_SUBSCRPTN?


currency


Ask a question What EDL objects are involved in the mappings for cif_notification_subscription?


EDL table: **cif_notification_subscription**  
EDL attributes: **subscribed**, **subscribed_cd**


Ask a question Generate a SQL query to join the EDL table for cif_notification_subscription with the relevant customer EDL table. Use the mapped attributes as the basis for the join.


The available data does not contain sufficient information to answer this question.


Ask a question exit
